# FCSG-Net training on Colab

Overflow path, reserved for the Phase 4 ablations so the Kaggle quota goes to
the main runs. Colab storage is ephemeral, so checkpoints go to Drive or they
are gone when the runtime recycles.

Runtime -> Change runtime type -> T4 GPU, first.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/fcsg-net/ckpt"   # survives the runtime dying
!mkdir -p "$OUT"

In [ ]:
import os, subprocess
REPO = "/content/fcsg-capstone"
if os.path.isdir(REPO):
    print(subprocess.run(["git", "-C", REPO, "pull"], capture_output=True, text=True).stdout)
else:
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/whynotramaa/fcsg-capstone", REPO], check=True)
os.chdir(REPO)

## Data

No attached datasets here, so fetch DIV2K over the network. About 3.3 GB and a
few minutes on Colab's link. Re-runs skip the download if the directory is
already there.

In [ ]:
%%bash
mkdir -p /content/DIV2K && cd /content/DIV2K
for split in train valid; do
  d=DIV2K_${split}_HR
  [ -d "$d" ] && { echo "$d present"; continue; }
  wget -q --show-progress https://data.vision.ee.ethz.ch/cvl/DIV2K/${d}.zip
  unzip -q ${d}.zip && rm ${d}.zip
done
ls -d /content/DIV2K/*/

## Tile cache

Decoding a 2K PNG for every sample held the Kaggle runs to 1.79 steps/s, which
is 46 hours for a 300k run. Build the cache once, then the loader is a memmap
slice and the GPU becomes the bottleneck again.

In [ ]:
!python training/build_tiles.py --data /content/DIV2K \
    --out /content/tiles.npy --patch 128 --per-image 64

### Does it actually help

2000 steps into a throwaway directory, so it does not resume from a real
checkpoint. Compare against the 1.79 steps/s the PNG loader managed.

In [ ]:
!rm -rf /content/timing
!python training/train.py --config configs/dncnn.toml --data /content/DIV2K \
    --tiles /content/tiles.npy --out /content/timing --steps 2000 --max-hours 0.5

import csv
r = [(int(x["step"]), float(x["secs"])) for x in
     csv.DictReader(open("/content/timing/train_log.csv")) if x["secs"]]
rate = (r[-1][0] - r[0][0]) / (r[-1][1] - r[0][1])
print(f"\n{rate:.2f} steps/s   (PNG loader managed 1.79)")
print(f"300k steps would now take {300000 / rate / 3600:.1f} h")

In [ ]:
# --max-hours matters here: the config default is 11.0, sized for Kaggle's 12h
# cap, and a free Colab runtime will not survive anywhere near that.
!python training/train.py --config configs/dncnn.toml --data /content/DIV2K \
    --tiles /content/tiles.npy --out "$OUT" --max-hours 3

In [ ]:
import glob
ckpt = sorted(glob.glob(OUT + "/ckpt_*.pt"))[-1]
!python evaluation/eval.py --ckpt $ckpt --data /content/DIV2K --out /content/results --limit 20
!python evaluation/plots.py --csv "$OUT/train_log.csv" --out /content/results/figures

## Publish the cache as a Kaggle Dataset

Uploading straight from Colab avoids a 2.5 GB round trip through a laptop.
Needs a `kaggle.json` from kaggle.com -> Settings -> Create New Token.
Afterwards every Kaggle run attaches this dataset instead of rebuilding.

In [ ]:
# Token comes from Colab Secrets (key icon, left sidebar), not a file. Add
# KAGGLE_USERNAME and KAGGLE_KEY from your kaggle.json, toggle notebook access
# on for both, and nothing lands on disk or in this .ipynb.
import json, os, shutil
from google.colab import userdata

os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")

!pip -q install kaggle

os.makedirs("/content/tilecache", exist_ok=True)
if os.path.exists("/content/tiles.npy"):
    shutil.move("/content/tiles.npy", "/content/tilecache/tiles.npy")
json.dump({"title": "DIV2K tiles 128px",
           "id": f"{os.environ['KAGGLE_USERNAME']}/div2k-tiles-128",
           "licenses": [{"name": "CC0-1.0"}]},
          open("/content/tilecache/dataset-metadata.json", "w"))

!kaggle datasets create -p /content/tilecache